# Teacher Labeling — GPT-4o Hop-1 Decompositions

**Goal.** Generate the training labels for the student. For each HotpotQA question I prompt
GPT-4o (the teacher) to produce the *first hop* of a reasoning decomposition as a small YAML
object: `thought`, `action`, `target_entity`. The student (Gemma-3-270M) will later learn to
imitate these.

HotpotQA ships with no reasoning traces — only questions, answers, and supporting facts. So the
teacher *synthesises* the decomposition the dataset never recorded; that synthesis is the whole
point of distillation here.

The teacher never sees the answer or the supporting facts. It works from the question alone — the
same input the student will see at inference time. I use the gold `supporting_facts` titles only
*afterwards*, to grade whether the teacher aimed at the right entity.

Everything in this notebook is plain HTTP calls to the API — no GPU needed.

## 1. Setup

The teacher is GPT-4o, so I need the OpenAI client and an API key. I read the key with `getpass`
rather than hard-coding it — the key is prompted at runtime and never written into the notebook or
its saved output, which matters since this notebook is public. It also behaves the same whether I
run on Colab, in VS Code, or locally.

In [2]:
!pip install -q openai pyyaml datasets

zsh:1: command not found: pip


In [1]:
import os
import getpass
from openai import OpenAI

# Prompt for the key once; it lives only in memory for this session, never in the notebook.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("[INFO] OpenAI client ready")

[INFO] OpenAI client ready


## 2. The teacher prompt

This is the instruction I give GPT-4o. It's a two-shot prompt: Example 1 is a *bridge* question
(look up the entity the question hangs on), Example 2 is a *comparison* question. For comparisons I
deliberately decompose one entity at a time — Hop-1 looks up the first entity and defers the actual
comparison to a later hop. That keeps every hop a single atomic action, so when a trajectory fails I
can localise which step broke.

The schema is fixed at three fields and `action` is always `Lookup`, which keeps the output trivial
to validate and grade.

In [2]:
TEACHER_PROMPT = """You are an analytical reasoning agent specialized in breaking down multi-hop questions.

Your objective is to determine ONLY the first logical step (Hop 1) required to solve the question.

CRITICAL CONSTRAINTS:
1. DO NOT answer the question. Stop reasoning immediately after formulating the first hop.
2. Output your response strictly in YAML format. Do not include introductory or concluding text, markdown formatting, or conversational filler.

YAML SCHEMA:
thought: [Your logical deduction of what needs to be found first]
action: [Lookup]
target_entity: [The exact entity string to target]

EXAMPLE 1:
Question: The Oberoi family is part of a hotel company that has a head office in what city?
thought: I need to find which hotel company the Oberoi family belongs to.
action: Lookup
target_entity: \"Oberoi family\"

EXAMPLE 2:
Question: Were Scott Derrickson and Ed Wood of the same nationality?
thought: I need to find the nationality of Scott Derrickson first to eventually compare it to Ed Wood.
action: Lookup
target_entity: \"Scott Derrickson\""""

print(TEACHER_PROMPT)

You are an analytical reasoning agent specialized in breaking down multi-hop questions.

Your objective is to determine ONLY the first logical step (Hop 1) required to solve the question.

CRITICAL CONSTRAINTS:
1. DO NOT answer the question. Stop reasoning immediately after formulating the first hop.
2. Output your response strictly in YAML format. Do not include introductory or concluding text, markdown formatting, or conversational filler.

YAML SCHEMA:
thought: [Your logical deduction of what needs to be found first]
action: [Lookup]
target_entity: [The exact entity string to target]

EXAMPLE 1:
Question: The Oberoi family is part of a hotel company that has a head office in what city?
thought: I need to find which hotel company the Oberoi family belongs to.
action: Lookup
target_entity: "Oberoi family"

EXAMPLE 2:
Question: Were Scott Derrickson and Ed Wood of the same nationality?
thought: I need to find the nationality of Scott Derrickson first to eventually compare it to Ed Wood

## 3. Pick a small set to label

I start small — 10 questions — before spending tokens on a full run. I pick a deliberate mix of 6
bridge and 4 comparison rather than just the first 10 rows, so both branches of the two-shot prompt
actually get exercised. Comparison questions are sparse near the front of the set, so I pull a larger
pool first and then sample by `type`.

In [3]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train[:200]")

bridge_rows = ds.filter(lambda r: r["type"] == "bridge").select(range(6))
comparison_rows = ds.filter(lambda r: r["type"] == "comparison").select(range(4))
label_set = concatenate_datasets([bridge_rows, comparison_rows])

print(f"label_set: {len(label_set)} rows "
      f"({sum(r['type']=='bridge' for r in label_set)} bridge, "
      f"{sum(r['type']=='comparison' for r in label_set)} comparison)\n")
for r in label_set:
    print(f"[{r['type']:10}] {r['question']}")

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

label_set: 10 rows (6 bridge, 4 comparison)

[bridge    ] The Oberoi family is part of a hotel company that has a head office in what city?
[bridge    ] Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
[bridge    ]  What nationality was James Henry Miller's wife?
[bridge    ] Cadmium Chloride is slightly soluble in this chemical, it is also called what?
[bridge    ] Which genus of moth in the world's seventh-largest country contains only one species?
[bridge    ] Who was once considered the best kick boxer in the world, however he has been involved in a number of controversies relating to his "unsportsmanlike conducts" in the sport and crimes of violence outside of the ring.
[comparison] Which magazine was started first Arthur's Magazine or First for Women?
[comparison] Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?
[comparison] Which band was founded first, Hole, the rock b

## 4. Ask the teacher for a decomposition

A small function: given a question, send the teacher prompt as the system message and the question
as the user message, call GPT-4o, and return the raw YAML string it produces.

In [8]:
def get_teacher_label(question:str) -> str:
    """Take a question string -> return the raw YAML string GPT-4o produces."""
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", 'content':TEACHER_PROMPT},
            {'role': 'user', 'content':f"Question: {question}"}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

In [9]:
print(get_teacher_label(label_set[0]['question']))

thought: I need to find which hotel company the Oberoi family belongs to.
action: Lookup
target_entity: "Oberoi family"


## Look around

In [ ]:
# repr will show the \ns and tell us definitively whether the string itself is doubled.
r = get_teacher_label(label_set[0]['question'])
print(repr(r))
print("len:", len(r))

In [9]:
print(label_set[2]['question'])
print("---")
print(get_teacher_label(label_set[2]['question']))

 What nationality was James Henry Miller's wife?
---
thought: I need to find out who James Henry Miller's wife was.
action: Lookup
target_entity: "James Henry Miller's wife"


In [10]:
print(label_set[2]['supporting_facts']['title'])

['Peggy Seeger', 'Peggy Seeger', 'Ewan MacColl']


In [66]:
label_set[2]['context']['title']

In [69]:
label_set[2]['context']['sentences'][5]

## 5. Parse and validate the output

The teacher returns a YAML string. I parse it and check it matches the schema exactly — the three
expected fields and `action == "Lookup"`. If the output is malformed I raise loudly rather than
silently dropping it, so bad traces stay visible during inspection.

In [10]:
import yaml

EXPECTED_FIELDS = {"thought", "action", "target_entity"}

def parse_and_validate(raw: str) -> dict:
    """Parse a teacher YAML string and validate the Hop-1 schema."""
    text = raw.strip()
    # Strip a stray ```yaml fence if the model wraps the output.
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("\n") + 1 :] if "\n" in text else text

    try:
        obj = yaml.safe_load(text)
    except yaml.YAMLError as e:
        raise ValueError(f"Not valid YAML: {e}\n---raw---\n{raw}")

    if not isinstance(obj, dict):
        raise ValueError(f"YAML did not parse to a mapping, got {type(obj)}:\n{raw}")
    if set(obj.keys()) != EXPECTED_FIELDS:
        raise ValueError(f"Field mismatch. Expected {EXPECTED_FIELDS}, got {set(obj.keys())}")
    if obj["action"] != "Lookup":
        raise ValueError(f"action must be 'Lookup', got {obj['action']!r}")

    return obj

In [7]:
output_1 = get_teacher_label(label_set[2]['question'])

## 6. Grade the target entity

This is the metric. The teacher never saw the supporting facts; here I use the gold supporting-fact
titles as an answer key and check whether the teacher's `target_entity` actually points at one of
them. How strict that match should be is the interesting design decision.

In [11]:
def grade_target_entity(target_entity: str, supporting_facts: list) -> str | None:
    """Lenient match of the teacher's target_entity against gold supporting_fact titles.
    Returns the matched title, or None - None means 'eyeball this one'.
    """
    pred = target_entity.strip().lower()
    if not pred:
        return False
    for title in {t.strip().lower() for t in supporting_facts}:
        if pred == title or pred in title or title in pred:
            return title

    return None

## 7. Run it and inspect

Wire the pieces together over the 10-row set and read every trace by hand. What I see here decides
the open questions: whether to keep `easy` rows, and how the teacher behaves when the Hop-1 entity
is described rather than named.

In [ ]:
hits = 0
for row in label_set:
    question = row['question']
    gold_titles = row['supporting_facts']['title'] # the title LiST, not the dict
    try:
        raw = get_teacher_label(question)
        parsed = parse_and_validate(raw)
        match = grade_target_entity(parsed['target_entity'], gold_titles)
    except Exception as e:
        print(f"[{row['type']:10}] {question}\n   !! FAILED: {e}\n")
        continue

    hit += bool(match)
    print(f"[{row['type']:10}] {question}")
    print(f"   target_entity :  {parsed['target_entity']!r}")
    print(f"   gold titles   :  {sorted(set(gold_titles))}")
    print(f"   match         :  {match if match else 'NO MATCH - eyeball'}")


print(f"String-match hits: {hits}/{len(label_set)} "
      f"(lower bound - aliases like Ewan MacColl surface here as false misses)")